# ORTHRUS-MSTC-PIDS — C8.2 Production Colab Notebook

This notebook is the production entry point for bounded-memory, restartable preprocessing and experiments. The paper configuration fits Word2Vec with `corpus_scope=train_only`; `official_full_dataset` is available only for original-ORTHRUS compatibility.

Runtime persistence model:

- The PostgreSQL server, apt packages, and `/content/orthrus` clone live in the temporary Colab VM and disappear after Runtime reset.
- Database dumps and preprocessing artifacts live in Google Drive and persist.
- Completion markers plus key-file validation allow each preprocessing stage to resume safely.
- `build_graphs` is primarily CPU RAM + PostgreSQL + Drive I/O; it does not require a GPU.
- Do not use “Run all”. Review the parameter cell, run setup cells in order, then explicitly enable one heavy stage.
- No training, preprocessing, restore, benchmark, or experiment matrix is enabled by default.


## 0. Unified parameters

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"

REPOSITORY_URL = "https://github.com/fish23611-beep/orthrus.git"
REPOSITORY_REF = "fix/c8-preprocess-correctness"
# Set to the final 40-character C8.2 commit for a strict frozen checkout.
EXPECTED_COMMIT = ""

DATASET = "THEIA_E3"
SEEDS = [0]
CORPUS_SCOPE = "train_only"  # paper: train_only; compatibility: official_full_dataset

DB_DUMPS = {
    "THEIA_E3": DATA_ROOT / "database_dumps/theia_e3.dump",
    "THEIA_E5": DATA_ROOT / "database_dumps/theia_e5.dump",
}
DATABASE_DUMP = DB_DUMPS[DATASET]

MOUNT_DRIVE = True
CLONE_OR_UPDATE = True
INSTALL_DEPENDENCIES = True
RESTORE_DATABASE = False
FORCE_DATABASE_RESTORE = False
RUN_PREPROCESS_BENCHMARK = False
RUN_PREPROCESS = False
FORCE_PREPROCESS = False
PREPROCESS_SUBSTAGES = "build_graphs,embed_nodes,embed_edges"
RUN_BASELINE_SMOKE = False
RUN_MAIN_MATRIX = False
RUN_ABLATIONS = False
RUN_COLLECT_EXPORT = False
RUN_DISPLAY_RESULTS = False
RUN_MANUAL_RESUME = False

EXPERIMENT_GROUP = "ablation"
RESUME_CONFIG = PROJECT_ROOT / "config/experiments/mstc_full.yml"
CHECKPOINT = Path("")

assert DATASET in DB_DUMPS
assert CORPUS_SCOPE in {"train_only", "official_full_dataset"}
print(f"Dataset={DATASET}; corpus_scope={CORPUS_SCOPE}; ref={REPOSITORY_REF}")
print("Heavy-stage switches are all disabled by default.")


## 1. Mount Drive and configure persistent roots

In [ ]:
import os

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab; Drive mount skipped.")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "database_dumps").mkdir(parents=True, exist_ok=True)
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
print(f"Persistent artifact root: {ARTIFACT_ROOT}")


## 2. Checkout one coherent repository ref

In [ ]:
import subprocess
import sys


def git(*args, capture=False, check=True):
    return subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), *args],
        check=check, text=True, capture_output=capture,
    )

if CLONE_OR_UPDATE:
    if (PROJECT_ROOT / ".git").is_dir():
        if git("status", "--porcelain", capture=True).stdout.strip():
            raise RuntimeError("Existing Colab clone is dirty; resolve it before checkout.")
    elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"Non-Git project directory is not empty: {PROJECT_ROOT}")
    else:
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

    git("fetch", "origin", "--tags", "--prune")
    remote_branch = git(
        "show-ref", "--verify", f"refs/remotes/origin/{REPOSITORY_REF}",
        capture=True, check=False,
    ).returncode == 0
    if remote_branch:
        git("checkout", REPOSITORY_REF)
        git("pull", "--ff-only", "origin", REPOSITORY_REF)
    else:
        git("checkout", REPOSITORY_REF)

actual_commit = git("rev-parse", "HEAD", capture=True).stdout.strip()
actual_ref_result = git("symbolic-ref", "--short", "-q", "HEAD", capture=True, check=False)
actual_ref = actual_ref_result.stdout.strip() or REPOSITORY_REF
if EXPECTED_COMMIT and actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Commit mismatch: expected {EXPECTED_COMMIT}, actual {actual_commit}")
if not EXPECTED_COMMIT:
    print("WARNING: EXPECTED_COMMIT is empty; pin it after the C8.2 commit is created.")
print(f"Actual ref: {actual_ref}")
print(f"Actual commit: {actual_commit}")

src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)


## 3. Resource preflight

In [ ]:
import shutil
import subprocess
import sys
import psutil
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
ram = psutil.virtual_memory()
print(f"CPU RAM: total={ram.total/1024**3:.1f} GB, available={ram.available/1024**3:.1f} GB")
for disk_path in (Path("/content"), DRIVE_ROOT):
    if disk_path.exists():
        usage = shutil.disk_usage(disk_path)
        print(f"{disk_path}: total={usage.total/1024**3:.1f} GB, free={usage.free/1024**3:.1f} GB")
print("build_graphs uses CPU RAM, PostgreSQL, and Drive I/O; GPU is not required.")
GPU_READY = torch.cuda.is_available()


## 4. Install Python dependencies

In [ ]:
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    packages = [
        "scikit-learn", "networkx", "xxhash", "graphviz", "psutil",
        "matplotlib", "wandb", "chardet", "nltk", "igraph", "cairocffi",
        "wget", "gensim", "pytz", "pandas", "yacs", "psycopg2-binary",
        "tqdm", "pyyaml", "torch_geometric",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
    torch_base = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_index = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
    for package in ("pyg_lib", "torch_scatter", "torch_sparse"):
        if subprocess.run([sys.executable, "-m", "pip", "show", package], capture_output=True).returncode:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package, "-f", wheel_index], check=True)


## 5. Import smoke test and environment record

In [ ]:
import importlib
import subprocess
import sys

for module_name in ("config", "orthrus", "torch", "torch_geometric", "pandas", "yaml"):
    module = importlib.import_module(module_name)
    print(f"import {module_name}: ok {getattr(module, '__version__', '')}")
environment_dir = ARTIFACT_ROOT / "environment"
environment_dir.mkdir(parents=True, exist_ok=True)
with (environment_dir / "pip_freeze.txt").open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True, text=True)


## 6. Resolve configuration and inspect persistent artifacts

In [ ]:
from config import get_runtime_required_args, get_yml_cfg
from pipeline_stages import check_all_preprocess_stages_complete, check_preprocess_stage_complete

PREPROCESS_CONFIG = PROJECT_ROOT / "config/orthrus.yml"


def fresh_preprocess_cfg():
    args = get_runtime_required_args(args=[
        DATASET, "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT), "--stages", "preprocess",
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}", "--skip-tracing",
    ])
    return get_yml_cfg(args)


def artifact_stats(path):
    path = Path(path)
    files = [item for item in path.rglob("*") if item.is_file()] if path.is_dir() else []
    total = sum(item.stat().st_size for item in files)
    return {"files": len(files), "MB": total / 1024**2, "GB": total / 1024**3}


def read_preprocess_status(current_cfg):
    status = {
        stage: check_preprocess_stage_complete(current_cfg, stage)
        for stage in ("build_graphs", "embed_nodes", "embed_edges", "metadata")
    }
    status["artifacts_complete"] = check_all_preprocess_stages_complete(current_cfg)
    return status


def print_status(label, status):
    print(label)
    for stage in ("build_graphs", "embed_nodes", "embed_edges", "metadata"):
        print(f"  {stage}: {'✓' if status[stage] else '✗'}")
    print(f"  artifacts_complete: {status['artifacts_complete']}")

cfg = fresh_preprocess_cfg()
status_before = read_preprocess_status(cfg)
print_status("Current:", status_before)
artifact_paths = {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}
for label, artifact_path in artifact_paths.items():
    stats = artifact_stats(artifact_path)
    print(f"{label}: files={stats['files']}, MB={stats['MB']:.2f}, GB={stats['GB']:.3f}")


## 7. PostgreSQL archive compatibility and restore

The helper first asks installed `pg_restore` clients to list the archive. Only if none can parse it does it configure PGDG and install the validated compatible major. It never deletes cluster data. A random password is set first through the Unix socket as the `postgres` system user; command display is redacted, then TCP authentication is verified.


In [ ]:
if not RESTORE_DATABASE:
    print("RESTORE_DATABASE=False; PostgreSQL is untouched.")
else:
    from colab_postgres import restore_database
    db_name = (
        cfg.dataset.database_all_file
        if cfg.graph_construction.build_graphs.use_all_files
        else cfg.dataset.database
    )
    credentials = restore_database(
        DATABASE_DUMP, db_name, force=FORCE_DATABASE_RESTORE
    )
    os.environ["ORTHRUS_DB_HOST"] = credentials["host"]
    os.environ["ORTHRUS_DB_PORT"] = credentials["port"]
    os.environ["ORTHRUS_DB_USER"] = credentials["user"]
    os.environ["ORTHRUS_DB_PASSWORD"] = credentials["password"]

    # Rebuild cfg after environment variables are set; stale cfg is forbidden.
    cfg = fresh_preprocess_cfg()
    assert cfg.database.host == "localhost"
    assert str(cfg.database.port) == "5432"
    assert cfg.database.user == "postgres"
    from provnet_utils import init_database_connection
    cursor, connection = init_database_connection(cfg)
    try:
        cursor.execute("SELECT 1 FROM event_table LIMIT 1;")
        print("ORTHRUS database preflight: ok")
    finally:
        cursor.close()
        connection.close()


## 8. Optional bounded-memory benchmark

In [ ]:
if not RUN_PREPROCESS_BENCHMARK:
    print("RUN_PREPROCESS_BENCHMARK=False; benchmark skipped.")
else:
    benchmark_command = [
        sys.executable, str(PROJECT_ROOT / "scripts/benchmark_preprocess_memory.py"),
        "--events", "200000", "--fetch-size", "8192",
    ]
    benchmark = subprocess.run(benchmark_command, cwd=PROJECT_ROOT, check=False)
    if benchmark.returncode:
        raise RuntimeError("Preprocessing benchmark failed; formal preprocessing is blocked.")


## 9. Restartable preprocessing

In [ ]:
cfg = fresh_preprocess_cfg()
before = read_preprocess_status(cfg)
print_status("Before:", before)

if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False; preprocessing skipped.")
else:
    command = [
        sys.executable, str(PROJECT_ROOT / "src/orthrus.py"), DATASET,
        "--config", str(PREPROCESS_CONFIG), "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "preprocess", "--preprocess-substages", PREPROCESS_SUBSTAGES,
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}", "--skip-tracing",
    ]
    if FORCE_PREPROCESS:
        command.append("--force-preprocess")
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

cfg = fresh_preprocess_cfg()
after = read_preprocess_status(cfg)
print_status("After:", after)
for label, artifact_path in {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}.items():
    stats = artifact_stats(artifact_path)
    print(f"{label}: files={stats['files']}, MB={stats['MB']:.2f}, GB={stats['GB']:.3f}")


---

## 10. Baseline Smoke Test

In [ ]:
import subprocess
import sys
import yaml
from pathlib import Path

if not RUN_BASELINE_SMOKE:
    print("RUN_BASELINE_SMOKE=False，跳过 baseline smoke test。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Baseline Smoke Test")
    print("=" * 60)

    BASE_CONFIG = PROJECT_ROOT / "config/experiments/baseline.yml"
    SMOKE_CONFIG = ARTIFACT_ROOT / "environment/smoke_baseline_1epoch.yml"

    smoke = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
    smoke.setdefault("pipeline", {})["run_tracing"] = False
    smoke.setdefault("detection", {}).setdefault("gnn_training", {})["num_epochs"] = 1
    SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    SMOKE_CONFIG.write_text(yaml.safe_dump(smoke, sort_keys=False), encoding="utf-8")

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"), "--dataset", DATASET, "--config", str(SMOKE_CONFIG), "--seed", str(SEEDS[0]), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate"]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Baseline smoke 通过")
    print("=" * 60)

---

## 11. 主模型矩阵

In [ ]:
import subprocess
import sys

if not RUN_MAIN_MATRIX:
    print("RUN_MAIN_MATRIX=False，跳过主模型矩阵。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("主模型矩阵")
    print("=" * 60)

    MAIN_CONFIGS = [PROJECT_ROOT / "config/experiments/baseline.yml", PROJECT_ROOT / "config/experiments/mstc_full.yml"]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, MAIN_CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("矩阵完成")
    print("=" * 60)

---

## 12. 消融与专项实验

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_ABLATIONS:
    print("RUN_ABLATIONS=False，跳过消融实验。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print(f"消融与专项实验: {EXPERIMENT_GROUP}")
    print("=" * 60)

    GROUPS = {
        "ablation": ["baseline.yml", "ablation_no_multiscale.yml", "ablation_no_gate.yml", "ablation_no_time.yml", "ablation_no_calibration.yml", "ablation_no_topk.yml", "mstc_full.yml"],
        "multiscale": ["multiscale_recent20.yml", "multiscale_recent24.yml", "multiscale_single_window.yml", "multiscale_equal.yml", "multiscale_gate.yml"],
        "time": ["time_type_only.yml", "time_time_only.yml", "time_joint.yml"],
        "calibration": ["calibration_max.yml", "calibration_quantile.yml", "calibration_kmeans.yml", "calibration_global_p.yml", "calibration_relation.yml", "calibration_hierarchical.yml"],
        "backbone": ["backbone_graphtransformer.yml", "backbone_graphsage_baseline.yml", "backbone_graphsage.yml", "backbone_mlp.yml"],
        "dataset_view": ["host_only.yml", "host_network_structure.yml", "host_network_full.yml"],
        "efficiency": ["baseline.yml", "efficiency_multiscale.yml", "efficiency_multiscale_time.yml", "mstc_full.yml"],
    }

    if EXPERIMENT_GROUP not in GROUPS:
        raise ValueError(f"未知组 {EXPERIMENT_GROUP!r}；可选 {sorted(GROUPS.keys())}")

    CONFIGS = [PROJECT_ROOT / "config/experiments" / name for name in GROUPS[EXPERIMENT_GROUP]]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("消融实验完成")
    print("=" * 60)

---

## 13. Checkpoint Resume

In [ ]:
import subprocess
import sys

if not RUN_MANUAL_RESUME:
    print("RUN_MANUAL_RESUME=False，未加载任何 checkpoint。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Checkpoint Resume")
    print("=" * 60)

    if not CHECKPOINT or not Path(CHECKPOINT).exists():
        raise FileNotFoundError(f"CHECKPOINT 不存在: {CHECKPOINT}")

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"), "--dataset", DATASET, "--config", str(RESUME_CONFIG), "--seed", str(SEEDS[0]), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate", "--checkpoint", str(CHECKPOINT)]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Resume 完成")
    print("=" * 60)

---

## 14. 结果收集与导出

In [ ]:
import subprocess
import sys

if not RUN_COLLECT_EXPORT:
    print("RUN_COLLECT_EXPORT=False，跳过结果收集与导出。")
else:
    print("=" * 60)
    print("结果收集与导出")
    print("=" * 60)

    collect_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/collect_results.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(collect_command, cwd=PROJECT_ROOT, check=True)

    export_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/export_tables.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(export_command, cwd=PROJECT_ROOT, check=True)

    print("结果收集与导出完成")
    print("=" * 60)

---

## 15. 结果展示

In [ ]:
from pathlib import Path
import pandas as pd

if not RUN_DISPLAY_RESULTS:
    print("RUN_DISPLAY_RESULTS=False，跳过结果展示。")
else:
    print("=" * 60)
    print("结果展示")
    print("=" * 60)

    RESULTS_ROOT = ARTIFACT_ROOT / "results"
    table_names = ["all_runs.csv", "main_results.csv", "ablation_results.csv", "calibration_results.csv", "efficiency_results.csv"]

    for name in table_names:
        path = RESULTS_ROOT / name
        if not path.is_file():
            print(f"警告: {name} 不存在")
            continue
        df = pd.read_csv(path)
        print(f"\n--- {name} ({len(df)} rows) ---")
        display(df)

    print("=" * 60)